# Importing the library and file

In [3]:
import pandas as pd
import numpy as np
np.random.seed(42)

In [4]:
df = pd.read_csv(r"C:\Users\Kurt_tuhin\Desktop\Snehashis\Projects\FinPay-Analysis\Data\Raw Data\Raw_data.csv")
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0


# Sampling and Formatting the dataset. (This is Fact Dataset in the start schema)

In [5]:
df.info()

df.describe()

df.head()

df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

In [6]:
df["type"].value_counts()                                                       #to see the payment types count


type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

In [7]:
df["isFraud"].value_counts(normalize=True)*100                                  #to see the if the fraud count is viable for sampling

isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64

In [8]:
                                                                                #the fraud proportion is too low(.12) so random sampling might not be useful for the analysis so we'll take different approach for sampling

fraud_df = df[df["isFraud"] == 1]                                               #splitting the dataset is fraud and non fraud part
nonfraud_df = df[df["isFraud"] == 0]

In [9]:
fraud_df = fraud_df.copy()                                                      #too safekeep the fraud case thats why different cell

In [10]:
target_size = 60000                                                             #Total sample size

nonfraud_sample = nonfraud_df.sample(
    n=target_size - len(fraud_df),
    random_state=42
)                                                                               #sampling the non_fraud data with sampling type 42 so that data can be reproducible

In [11]:
df_sample = pd.concat(
    [fraud_df, nonfraud_sample],
    ignore_index=True
)                                                                               #combibining the dataset of fraud and non_fraud

In [12]:
df_sample = df_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)                                                        #shuffling the data to avoid the grouping and bias

In [13]:
df_sample["isFraud"].value_counts(normalize=True)*100                           #now the fraud_cases are viable ~13% so we can move on with this data

isFraud
0    86.311667
1    13.688333
Name: proportion, dtype: float64

In [14]:
df_sample["TransactionID"] = [
    f"TXN{i:06d}" for i in range(1, len(df_sample) + 1)
]                                                                               #adding the unique transaction for the dataset which works as primary key for this data set

cols = ["TransactionID"] + [c for c in df_sample.columns if c != "TransactionID"]
df_sample = df_sample[cols]                                                     #moving the TransactionID column to the front

In [15]:
df_sample

,TransactionID,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,TXN000001,15,CASH_OUT,34492.73,C2065411884,0.00,0.00,C83851825,275281.09,475352.85,0,0
1,TXN000002,347,CASH_OUT,244530.83,C2090567497,0.00,0.00,C1373622639,913985.65,1158516.48,0,0
2,TXN000003,305,PAYMENT,17523.48,C241055782,49957.00,32433.52,M1431688274,0.00,0.00,0,0
3,TXN000004,249,PAYMENT,17555.22,C171525442,0.00,0.00,M1535177085,0.00,0.00,0,0
4,TXN000005,352,CASH_OUT,84782.10,C585202294,105294.00,20511.90,C610837419,325520.54,410302.64,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
59995,TXN059996,184,CASH_OUT,80106.92,C268957473,0.00,0.00,C1549796269,244140.55,324247.47,0,0
59996,TXN059997,353,CASH_OUT,54671.54,C386772449,0.00,0.00,C573931881,7006822.11,7061493.65,0,0
59997,TXN059998,71,TRANSFER,1871553.73,C677678546,1871553.73,0.00,C646623849,0.00,0.00,1,0
59998,TXN059999,335,CASH_IN,98932.38,C2033626434,1609496.52,1708428.90,C1360574838,3484293.51,3385361.13,0,0


In [16]:
bins = [-1,1000,10000,50000,float("inf")]                                       #categorising the transaction amount for better visual analysis. We didnt use if here cause if in
                                                                                #future we want to add new value in labels we can just update bins and labels instead of whole function
                                                                                #bins probably start at 0, and there are transactions with amount = 0 or values outside the bin edges. so started the bin range with -1

labels = [
    "Small",
    "Medium",
    "Large",
    "Very Large"
]

df_sample["AmountBand"] = pd.cut(
    df_sample["amount"],
    bins=bins,
    labels=labels
)

In [17]:
fee_rate = {                                                                    #transaction fees as the only source of revenue for the company
    "PAYMENT":0.003,
    "TRANSFER":0.004,
    "CASH_OUT":0.002,
    "CASH_IN":0.0015,
    "DEBIT":0.0025
}

df_sample["Fee"] = (
    df_sample["amount"] *
    df_sample["type"].map(fee_rate)
).round(2)

In [18]:
df_sample["Cashback"] = 0.0                                                     #cashback system for the payment type of transaction. Used flot type to avoid future variable type erros

mask = df_sample["type"]=="PAYMENT"

df_sample.loc[mask,"Cashback"] = (
    df_sample.loc[mask,"amount"]*0.005
).clip(upper=100)

In [19]:
df_sample["NetRevenue"] = (                                                     #net revenue calculation
    df_sample["Fee"] -
    df_sample["Cashback"]
).round(2)

In [20]:
df_sample["HighValue"] = (                                                      #high value transaction calculation as completely distiguised table as its a important KPI
    df_sample["amount"]>50000
)

In [21]:
df_sample["BalanceReduction"] = (                                               #balance reduction calculation
    df_sample["oldbalanceOrg"]-
    df_sample["newbalanceOrig"]
)

In [22]:
df_sample.info()

<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   TransactionID     60000 non-null  str     
 1   step              60000 non-null  int64   
 2   type              60000 non-null  str     
 3   amount            60000 non-null  float64 
 4   nameOrig          60000 non-null  str     
 5   oldbalanceOrg     60000 non-null  float64 
 6   newbalanceOrig    60000 non-null  float64 
 7   nameDest          60000 non-null  str     
 8   oldbalanceDest    60000 non-null  float64 
 9   newbalanceDest    60000 non-null  float64 
 10  isFraud           60000 non-null  int64   
 11  isFlaggedFraud    60000 non-null  int64   
 12  AmountBand        60000 non-null  category
 13  Fee               60000 non-null  float64 
 14  Cashback          60000 non-null  float64 
 15  NetRevenue        60000 non-null  float64 
 16  HighValue         60000 non-null 

In [23]:
df_sample.to_csv(                                                               #exporting the dataset
    "Fact_Transactions.csv",
    index=False
)

# Creating the Customer Dimension Tables


In [24]:
num_customers = 20000                                                           #generating the customersID cause the customer ID in the transaction table are all unique and is not useful for customer analysis

dim_customer = pd.DataFrame({
    "CustomerKey": range(1, num_customers + 1),
    "CustomerID": [f"CUST{str(i).zfill(5)}" for i in range(1, num_customers + 1)]
})

dim_customer.head()

,CustomerKey,CustomerID
0,1,CUST00001
1,2,CUST00002
2,3,CUST00003
3,4,CUST00004
4,5,CUST00005


In [25]:
segments = ["Standard", "Gold", "Premium"]                                      #customer segment generation
segment_prob = [0.65, 0.25, 0.10]

dim_customer["CustomerSegment"] = np.random.choice(
    segments,
    size=num_customers,
    p=segment_prob
)

In [26]:
states = [                                                                      #limiting the fintech market to some states only
    "Maharashtra",
    "Karnataka",
    "Delhi",
    "Tamil Nadu",
    "Gujarat",
    "Telangana",
    "West Bengal",
    "Uttar Pradesh",
    "Kerala",
    "Rajasthan"
]

state_prob = [
    0.18,
    0.12,
    0.10,
    0.09,
    0.08,
    0.08,
    0.08,
    0.10,
    0.08,
    0.09
]

dim_customer["State"] = np.random.choice(
    states,
    size=num_customers,
    p=state_prob
)

In [27]:
city_dict = {                                                                   #assigning the city districts
    "Maharashtra": ["Mumbai", "Pune", "Nagpur"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangaluru"],
    "Delhi": ["New Delhi"],
    "Tamil Nadu": ["Chennai", "Coimbatore", "Madurai"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara"],
    "Telangana": ["Hyderabad", "Warangal"],
    "West Bengal": ["Kolkata", "Siliguri", "Durgapur"],
    "Uttar Pradesh": ["Lucknow", "Noida", "Kanpur"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode"],
    "Rajasthan": ["Jaipur", "Jodhpur", "Udaipur"]
}

dim_customer["City"] = dim_customer["State"].apply(
    lambda x: np.random.choice(city_dict[x])
)

In [28]:
gender = ["Male", "Female"]                                                     #assigning the genders

gender_prob = [0.58, 0.42]

dim_customer["Gender"] = np.random.choice(
    gender,
    size=num_customers,
    p=gender_prob
)

In [29]:
age_groups = [                                                                  #assigning the age groups
    "18-25",
    "26-35",
    "36-45",
    "46-60",
    "60+"
]

age_prob = [
    0.22,
    0.36,
    0.24,
    0.13,
    0.05
]

dim_customer["AgeGroup"] = np.random.choice(
    age_groups,
    size=num_customers,
    p=age_prob
)

In [30]:
income = [                                                                      #assigning the income group
    "Low",
    "Middle",
    "High"
]

income_prob = [
    0.35,
    0.50,
    0.15
]

dim_customer["IncomeGroup"] = np.random.choice(
    income,
    size=num_customers,
    p=income_prob
)

In [31]:
kyc = [                                                                         #assigning the KYC status
    "Verified",
    "Pending",
    "Rejected"
]

kyc_prob = [
    0.96,
    0.03,
    0.01
]

dim_customer["KYCStatus"] = np.random.choice(
    kyc,
    size=num_customers,
    p=kyc_prob
)

In [32]:
join_years = [                                                                  #assigning the join years
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
]

join_prob = [
    0.04,
    0.06,
    0.10,
    0.12,
    0.14,
    0.16,
    0.18,
    0.20
]

dim_customer["JoinYear"] = np.random.choice(
    join_years,
    size=num_customers,
    p=join_prob
)

In [33]:
dim_customer["CustomerTenure"] = 2026 - dim_customer["JoinYear"]                #assigning the customer tenure by year

In [34]:
dim_customer.tail()

,CustomerKey,CustomerID,CustomerSegment,State,City,Gender,AgeGroup,IncomeGroup,KYCStatus,JoinYear,CustomerTenure
19995,19996,CUST19996,Gold,Uttar Pradesh,Kanpur,Male,18-25,Low,Verified,2024,2
19996,19997,CUST19997,Standard,Uttar Pradesh,Lucknow,Female,26-35,Low,Verified,2021,5
19997,19998,CUST19998,Standard,Karnataka,Mysuru,Male,18-25,Low,Verified,2020,6
19998,19999,CUST19999,Standard,Tamil Nadu,Madurai,Male,46-60,Low,Verified,2025,1
19999,20000,CUST20000,Standard,Tamil Nadu,Coimbatore,Male,46-60,Low,Verified,2019,7


In [35]:
dim_customer.to_csv(                                                            #exporting the dataset
    "dim_customer.csv",
    index=False
)

# Replacing the customerID to the transaction table with the new customer ID

In [36]:
customer_weights = (                                                            #assigning the customer segments a weightage to map to the transaction table
    dim_customer["CustomerSegment"]                                             #so that the high value transaction can be mapped to gold or premium customers
    .map({
        "Standard": 1,
        "Gold": 2,
        "Premium": 4
    })
)

In [37]:
df_sample["CustomerKey"] = np.random.choice(                                    #replacing the old customerID column with the new one
    dim_customer["CustomerKey"],                                                #mapping the CustomerKey and not CustomerId as if the naming convention changes, the customer table will still be mapped
    size=len(df_sample),
    replace=True,
    p=customer_weights / customer_weights.sum()
)

df_sample.drop(columns=["nameOrig"], inplace=True)


#Creating the Merchant Dimension Tables

In [38]:
num_merchants = 8000                                                            #creating the merchant table to assign all the merchant attributes

dim_merchant = pd.DataFrame({
    "MerchantKey": range(1, num_merchants + 1),
    "MerchantID": [f"MER{str(i).zfill(5)}" for i in range(1, num_merchants + 1)]
})

dim_merchant.head()

,MerchantKey,MerchantID
0,1,MER00001
1,2,MER00002
2,3,MER00003
3,4,MER00004
4,5,MER00005


In [39]:
merchant_categories = [                                                         #assigning the merchant categories
    "Retail",
    "E-Commerce",
    "Food & Dining",
    "Fuel",
    "Healthcare",
    "Travel",
    "Entertainment",
    "Education",
    "Utilities",
    "Government"
]

category_prob = [
    0.22,
    0.18,
    0.15,
    0.08,
    0.08,
    0.07,
    0.06,
    0.05,
    0.07,
    0.04
]

dim_merchant["MerchantCategory"] = np.random.choice(
    merchant_categories,
    size=num_merchants,
    p=category_prob
)

In [40]:
business_size = [                                                               #assigning the business size for the merchants
    "Small",
    "Medium",
    "Enterprise"
]

size_prob = [
    0.65,
    0.25,
    0.10
]

dim_merchant["BusinessSize"] = np.random.choice(
    business_size,
    size=num_merchants,
    p=size_prob
)

In [41]:
states = [                                                                      #limiting the fintech market to some states only
    "Maharashtra",
    "Karnataka",
    "Delhi",
    "Tamil Nadu",
    "Gujarat",
    "Telangana",
    "West Bengal",
    "Uttar Pradesh",
    "Kerala",
    "Rajasthan"
]

state_prob = [
    0.18,
    0.12,
    0.10,
    0.09,
    0.08,
    0.08,
    0.08,
    0.10,
    0.08,
    0.09
]

dim_merchant["State"] = np.random.choice(
    states,
    size=num_merchants,
    p=state_prob
)

In [42]:
city_dict = {                                                                   #assigning the city to the merchants
    "Maharashtra": ["Mumbai", "Pune", "Nagpur"],
    "Karnataka": ["Bengaluru", "Mysuru", "Mangaluru"],
    "Delhi": ["New Delhi"],
    "Tamil Nadu": ["Chennai", "Coimbatore", "Madurai"],
    "Gujarat": ["Ahmedabad", "Surat", "Vadodara"],
    "Telangana": ["Hyderabad", "Warangal"],
    "West Bengal": ["Kolkata", "Siliguri", "Durgapur"],
    "Uttar Pradesh": ["Lucknow", "Noida", "Kanpur"],
    "Kerala": ["Kochi", "Thiruvananthapuram", "Kozhikode"],
    "Rajasthan": ["Jaipur", "Jodhpur", "Udaipur"]
}

dim_merchant["City"] = dim_merchant["State"].apply(
    lambda x: np.random.choice(city_dict[x])
)

In [43]:
settlement = [                                                                  #assigning the settlement time
    "T+1",
    "T+2",
    "T+3"
]

settlement_prob = [
    0.70,
    0.25,
    0.05
]

dim_merchant["SettlementCycle"] = np.random.choice(
    settlement,
    size=num_merchants,
    p=settlement_prob
)

In [44]:
partner_year = [                                                                #assigning the year from when the merchants are added to the platform
    2019,
    2020,
    2021,
    2022,
    2023,
    2024,
    2025,
    2026
]

partner_prob = [
    0.04,
    0.06,
    0.10,
    0.12,
    0.13,
    0.17,
    0.19,
    0.19
]

dim_merchant["PartnerSince"] = np.random.choice(
    partner_year,
    size=num_merchants,
    p=partner_prob
)

In [45]:
dim_merchant["MerchantTenure"] = (                                              #calculating the merchant tenure to current year
    2026 - dim_merchant["PartnerSince"]
)

In [46]:
dim_merchant.head()

,MerchantKey,MerchantID,MerchantCategory,BusinessSize,State,City,SettlementCycle,PartnerSince,MerchantTenure
0,1,MER00001,Travel,Small,Kerala,Kozhikode,T+1,2021,5
1,2,MER00002,E-Commerce,Small,Maharashtra,Nagpur,T+1,2024,2
2,3,MER00003,Government,Small,Maharashtra,Pune,T+2,2023,3
3,4,MER00004,Food & Dining,Small,Uttar Pradesh,Kanpur,T+1,2023,3
4,5,MER00005,Retail,Medium,Delhi,New Delhi,T+1,2025,1


In [47]:
dim_merchant.to_csv(                                                            #exporting the dataset
    "dim_merchant.csv",
    index=False
)

# Replacing the merchantID to the transaction table with the new merchantID

In [48]:
merchant_weights = (
    dim_merchant["BusinessSize"]
    .map({
        "Small": 1,
        "Medium": 3,
        "Enterprise": 8
    })
)

In [49]:
df_sample["MerchantKey"] = np.random.choice(
    dim_merchant["MerchantKey"],
    size=len(df_sample),
    replace=True,
    p=merchant_weights / merchant_weights.sum()
)

In [50]:
df_sample.drop(columns=["nameDest"], inplace=True)

In [51]:
df_sample.head()

,TransactionID,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,AmountBand,Fee,Cashback,NetRevenue,HighValue,BalanceReduction,CustomerKey,MerchantKey
0,TXN000001,15,CASH_OUT,34492.73,0.0,0.00,275281.09,475352.85,0,0,Large,68.99,0.0000,68.99,False,0.00,5091,3531
1,TXN000002,347,CASH_OUT,244530.83,0.0,0.00,913985.65,1158516.48,0,0,Very Large,489.06,0.0000,489.06,True,0.00,14465,3251
2,TXN000003,305,PAYMENT,17523.48,49957.0,32433.52,0.00,0.00,0,0,Large,52.57,87.6174,-35.05,False,17523.48,7818,17
3,TXN000004,249,PAYMENT,17555.22,0.0,0.00,0.00,0.00,0,0,Large,52.67,87.7761,-35.11,False,0.00,7272,7796
4,TXN000005,352,CASH_OUT,84782.10,105294.0,20511.90,325520.54,410302.64,0,0,Very Large,169.56,0.0000,169.56,True,84782.10,15893,6154


# Creating Date Dimension


In [53]:
num_steps = df_sample["step"].max()


start_date = pd.Timestamp("2026-01-01 00:00:00")                                #creating the time stamp for the transaction

dim_date = pd.DataFrame({
    "step": range(1, num_steps + 1)
})

dim_date["DateTime"] = (
    start_date +
    pd.to_timedelta(dim_date["step"] - 1, unit="h")
)

In [54]:
dim_date["Date"] = dim_date["DateTime"].dt.date                                 #assigning the Date, Year, Month, Quarter, Weekday, Day, Hour according to the timestamp
dim_date["Year"] = dim_date["DateTime"].dt.year
dim_date["Month"] = dim_date["DateTime"].dt.month_name()
dim_date["MonthNo"] = dim_date["DateTime"].dt.month
dim_date["Quarter"] = dim_date["DateTime"].dt.quarter
dim_date["Weekday"] = dim_date["DateTime"].dt.day_name()
dim_date["Day"] = dim_date["DateTime"].dt.day
dim_date["Hour"] = dim_date["DateTime"].dt.hour

In [55]:
dim_date["Weekend"] = np.where(                                                 #assigning the weekend and weekday
    dim_date["Weekday"].isin(["Saturday", "Sunday"]),
    "Weekend",
    "Weekday"
)

In [56]:
def time_bucket(hour):                                                          #assigning the time bucket according to the hour
    if 0 <= hour < 6:
        return "Night"
    elif 6 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 18:
        return "Afternoon"
    else:
        return "Evening"

dim_date["TimeBucket"] = dim_date["Hour"].apply(time_bucket)

In [57]:
dim_date["DateKey"] = (                                                         #assigning the date key
    dim_date["DateTime"]
    .dt.strftime("%Y%m%d%H")
    .astype(int)
)

In [58]:
dim_date = dim_date[                                                            #rearranging the date table
    [
        "DateKey",
        "step",
        "DateTime",
        "Date",
        "Year",
        "Quarter",
        "MonthNo",
        "Month",
        "Day",
        "Weekday",
        "Weekend",
        "Hour",
        "TimeBucket"
    ]
]

In [59]:
dim_date.to_csv(                                                                #exporting the dataset
    "dim_date.csv",
    index=False
)